# 5. 深度学习计算

## 5.1 层和块

> 之前的代码中，我们仅使用过 `nn.Sequential` 这个线性容器 (无法处理条件分支逻辑)
>
> 为了实现更复杂的架构，我们需要学习 PyTorch 的核心基类：<b>nn.Module<b>。

### 1. 核心概念：块 (Blocks)

在 PyTorch 中，**“块”**（Block）可以描述单个层、由多个层组成的组件，甚至整个模型本身。
*   **数学本质**：块是一个将输入张量转化为输出张量的函数。
*   **工程本质**：块是一个继承自 `nn.Module` 的类，它维护着**状态**（参数）和**行为**（前向传播）。

---


### 2. 自定义多层感知机块

In [6]:
import torch
from torch import nn, Tensor

class MLP(nn.Module):
    """自定义多层感知机块。
    
    展示了 nn.Module 的基本结构：__init__ 负责定义组件，forward 负责定义计算流。
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        """初始化模型层。
        
        Args:
            input_dim: 输入特征维度。
            hidden_dim: 隐藏层神经元数量。
            output_dim: 输出类别数量。
        """
        # 必须调用父类的 __init__(), 否则无法自动注册参数
        super().__init__()

        # 将层定义为类的成员属性
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, X: Tensor) -> Tensor:
        """定义前向传播逻辑。
        
        Args:
            X: 输入向量。

        Returns:
            Tensor: 输出向量。
        """
        # 我们可以自由控制计算顺序，甚至可以在这里加入 Python 的 if/for 逻辑
        h = self.relu(self.hidden(X))
        return self.out(h)
    
# --- 使用示例 ---
net = MLP(input_dim=20, hidden_dim=256, output_dim=10)
X = torch.randn(2, 20)
# nn.Module 的魔法方法 __call__ 允许调用 net(X) 时执行 forward 逻辑
# 建议：始终使用 net(X) 而不是 net.forward(X), 因为直接调用 forward 可能会导致某些监控或调试工具失效。
output = net(X) 
print(f"自定义的 MLP 输出形状: {output.shape}")

自定义的 MLP 输出形状: torch.Size([2, 10])


---

### 3. 细致梳理：`nn.Module` 

为什么继承 `nn.Module` 后，模型就能自动求导并管理参数了？

1.  **参数注册 (Parameter Registration)**：
    当你执行 `self.hidden = nn.Linear(...)` 时，`nn.Module` 会在后台自动识别出这是一个包含参数的层，并将其权重和偏置加入到模型的 `parameters()` 列表中。
2.  **`__call__` 与 `forward`**：
    在 PyTorch 中，你调用 `net(X)` 而不是 `net.forward(X)`。这是因为父类实现了 `__call__` 方法，它在运行 `forward` 之前和之后会执行一些必要的“钩子”操作（如注册梯度）。
3.  **递归性**：
    一个 `nn.Module` 可以包含另一个 `nn.Module`。这允许我们像搭乐高一样，把小组件拼成大系统。

---


### 4. 实现一个自定义的顺序块 (Sequential)

In [7]:
class MySequential(nn.Module):
    """手动实现 Sequential 容器。"""
    def __init__(self, *args: nn.Module):
        super().__init__()
        # _modules 是父类 nn.Module 内部维护的一个有序字典
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X: Tensor) -> Tensor:
        # 按顺序执行每一个已注册的块
        for block in self._modules.values():
            X = block(X)
        return X
    
# --- 使用方法 ---
net_test = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
print(f"MySequential 输出形状: {net_test(torch.randn(2, 20)).shape}")

MySequential 输出形状: torch.Size([2, 10])


---

### 5. 在前向传播中执行复杂计算 (Fixed Parameters)

> 这是 nn.Sequential 无法做到的事：在计算过程中加入不参与训练的常数或复杂的控制流。


In [8]:
class FixedMLP(nn.Module):
    """展示在 forward 中加入复杂计算逻辑。"""
    def __init__(self):
        super().__init__()
        # rand_weight 是随机生成的，且不设为 Parameter, 因此不会被梯度更新
        self.rand_weight = torch.randn(20, 20)
        self.linear = nn.Linear(20, 20)

    def forward(self, X: Tensor) -> Tensor:
        X = self.linear(X)
        # 使用常数权重进行计算，并加入复杂的 Python 控制流
        X = torch.relu(X @ self.rand_weight + 1)

        # 复用同一个线性层 (参数共享)
        X = self.linear(X)

        # 控制流演示
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

# --- 使用方法 ---
net_current = FixedMLP()
X = torch.randn(2, 20)
output = net_current(X)

print(f"1. 输出结果: {output.item():.4f} (是一个标量)")

# 演示 1：证明 rand_weight 不在参数列表中（不会被训练）
params = [name for name, _ in net_current.named_parameters()]
print(f"2. 模型参数列表: {params}")
print(f"   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter")

# 演示 2：执行反向传播，查看梯度
output.backward()
print(f"3. linear.weight 的梯度形状: {net_current.linear.weight.grad.shape}")

# 演示 3：由于 forward 中有 while 循环，我们可以检查输出是否符合预期（sum < 1）
print(f"4. 最终 X.sum() 是否小于 1: {output < 1}")

1. 输出结果: -0.0828 (是一个标量)
2. 模型参数列表: ['linear.weight', 'linear.bias']
   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter
3. linear.weight 的梯度形状: torch.Size([20, 20])
4. 最终 X.sum() 是否小于 1: True


In [9]:
# 嵌套块练习
class NestMLP(nn.Module):
    """嵌套块练习：级联(串联)两个 MLP。"""
    def __init__(self):
        super().__init__()
        # 在容器中嵌套我们自定义的 MLP 块
        self.net = nn.Sequential(
            MLP(20, 64, 32),
            MLP(32, 16, 10)
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
# --- 测试 ---
nest_net = NestMLP()
print(f"NestMLP 输出形状: {nest_net(torch.randn(2, 20)).shape}")

NestMLP 输出形状: torch.Size([2, 10])


### 6. 关于“块”的工程规范建议

**`nn.Module`** 的开发准则：

1.  **参数注册**：永远在 `__init__` 中将子模块赋值给 `self.xxx`，这样 `net.parameters()` 才能找到它们。
2.  **逻辑位置**：所有的参数初始化逻辑放在 `__init__`，所有的计算逻辑放在 `forward`。**严禁**在 `forward` 中定义新的 `nn.Linear`（否则每次运行都会创建新权重）。
3.  **类型注解**：在 `forward` 中明确标注 `Tensor` 的输入输出，这对于大型模型 Debug 至关重要。

---


## 5.2 参数管理

### 1. 参数访问

当我们使用 nn.Sequential 或自定义 nn.Module 组建网络后，参数被组织在一个嵌套的层次结构中。

#### 1.1 访问特定层的参数

我们可以通过索引或属性名来获取特定层的 state_dict（状态字典）。

In [10]:
import torch
from torch import nn, Tensor

def inspect_parameters(net: nn.Module) -> None:
    """演示如何访问模型不同层的参数。"""

    # 1. 访问特定层的参数
    # net[2] 表示访问 Sequential 中的第三层 (0-based)
    print("Layer 2 State Dict Keys:", net[2].state_dict().keys())

    # 2. 访问具体的权重对象
    # weight 是一个 nn.Parameter 对象
    weight_param = net[2].weight
    print(f"权重类型: {type(weight_param)}")
    print(f"权重数值 (Tensor):\n{weight_param.data}")
    print(f"权重梯度: {weight_param.grad}") # 训练前为 None

# --- 示例 ---
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.randn(2, 4)
net(X) # 运行一次前向传播以确保参数被初始化

inspect_parameters(net)

Layer 2 State Dict Keys: odict_keys(['weight', 'bias'])
权重类型: <class 'torch.nn.parameter.Parameter'>
权重数值 (Tensor):
tensor([[-0.0142,  0.1374,  0.0908, -0.1526,  0.2717, -0.0978, -0.0806,  0.0403]])
权重梯度: None


#### 1.2 一次性访问所有参数

我们常用 named_parameters() 来遍历整个网络的参数名和数值。

In [11]:
def print_all_params(net: nn.Module) -> None:
    """遍历并打印网络中所有参数的名称和形状。"""
    for name, param in net.named_parameters():
        print(f"参数名: {name:20} | 形状: {list(param.shape)}")

# 演示
print_all_params(net)

参数名: 0.weight             | 形状: [8, 4]
参数名: 0.bias               | 形状: [8]
参数名: 2.weight             | 形状: [1, 8]
参数名: 2.bias               | 形状: [1]


#### 1.3 从嵌套块搜集参数

In [12]:
# 生成块的函数
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在此处嵌套
        net.add_module(f"block {i}", block1())
    return net

# --- 示例 ---
nest_net = nn.Sequential(block2(), nn.Linear(4, 1))
X = torch.randn(2, 4)
nest_net(X)

print(nest_net)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [13]:
# 访问第一个块的，第二个子块的第一层的偏置项 (0-based)
nest_net[0][1][0].bias

Parameter containing:
tensor([ 0.2203,  0.2943,  0.4953, -0.1088,  0.3055, -0.1241, -0.4215,  0.1861],
       requires_grad=True)

---

### 2. 参数初始化

PyTorch 默认会进行初始化，但为了数值稳定性（4.8节内容），我们通常需要自定义初始化策略。


#### 2.1 使用内置初始化器


In [14]:
def init_normal(m: nn.Module) -> None:
    """内置正态分布初始化策略。"""
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def init_constant(m: nn.Module) -> None:
    """内置常数初始化策略。"""
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# --- 使用方法 ---
net.apply(init_normal) # 递归应用
net.apply(init_constant)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

#### 2.2 自定义初始化逻辑

如果教材要求的逻辑非常特殊（比如：$w \sim \mathcal{U}(-10, 10)$ 且绝对值大于 5 才保留），我们可以直接操作 `.data`。


In [15]:
def my_custom_init(m: nn.Module) -> None:
    """实现复杂的自定义初始化逻辑。"""
    if isinstance(m, nn.Linear):
        print(f"正在初始化层: {m}")
        # 1. 均匀分布
        nn.init.uniform_(m.weight, -10, 10)
        # 2. 复杂的逻辑过滤：只保留绝对值 >= 5 的元素，其余置 0
        m.weight.data *= (m.weight.data.abs() >= 5)

# --- 示例 ---
net.apply(my_custom_init)

正在初始化层: Linear(in_features=4, out_features=8, bias=True)
正在初始化层: Linear(in_features=8, out_features=1, bias=True)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

##### **注意**：

1. `.data` 是 PyTorch 的历史遗留属性，即使在初始化时使用 `.data` 是“安全”的，但它存在 **“静默错误”**：

    如果你习惯了使用 `.data`，万一在 `forward` 过程中误用了它（例如 `x.data += 1`），PyTorch 的自动求导系统**完全不会报错**，但最终算出来的梯度会是错的。这种 Bug 极难排查。


2. 初始化参数的现代标准做法：

    虽然 `.data` 在初始化时有效，但现在 PyTorch 官方建议使用 `torch.no_grad()`，因为它更显式且安全：

In [16]:
# 现代化的 PyTorch 自定义初始化
@torch.no_grad
def my_custom_init(m: nn.Module) -> None:
    """实现复杂的自定义初始化逻辑。"""
    if isinstance(m, nn.Linear):
        print(f"正在初始化层: {m}")
        # 1. 均匀分布
        nn.init.uniform_(m.weight, -10, 10)
        # 2. 复杂的逻辑过滤：只保留绝对值 >= 5 的元素，其余置 0
        # 直接对 Tensor 进行原位操作，不需要访问 .data
        m.weight *= (m.weight.abs() >= 5)

# --- 示例 ---
net.apply(my_custom_init)

正在初始化层: Linear(in_features=4, out_features=8, bias=True)
正在初始化层: Linear(in_features=8, out_features=1, bias=True)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

---

### 3. 参数绑定与共享

有时我们希望在模型的不同层之间共享同一组权重。这在循环神经网络（RNN）或自编码器（Autoencoder）中非常常见。


#### 3.1 代码实现：同一个实例多次调用

实现共享参数最简单的方法是：先定义一个层实例，然后在 `forward` 中多次调用它。

> 实际上 5.1 中有演示过共享参数的情况。


In [17]:
class SharedParamMLP(nn.Module):
    """展示如何在不同层之间共享参数。"""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        # 定义一个共享层
        self.shared = nn.Linear(hidden_dim, hidden_dim)

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            self.shared, # 第一次使用
            nn.ReLU(),
            self.shared, # 第二次使用（参数与上面完全绑定）
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
# --- 验证绑定 ---
net_shared = SharedParamMLP(4, 8, 1)
# 检查 Layer 2 和 Layer 4 的参数是否指向同一个内存地址
print(f"参数是否物理共享: {net_shared.net[2].weight is net_shared.net[4].weight}")

参数是否物理共享: True


**注意**: 由于它们指向同一个内存地址，在反向传播时，这个共享层的梯度是两次调用产生的梯度之和。

##### **⚠️ 重要说明：关于代码规范的注意**

> **注意**：上述代码中 `self.shared` 层既作为独立属性定义，又被放入 `nn.Sequential` 容器的做法，**仅为了直观演示“物理内存共享”这一概念**。在实际生产环境和科研工程中，这种写法被视为 **“Bad Practice”（不佳实践）**。

**原因如下：**
1.  **重复注册与命名混乱**：PyTorch 会为同一个参数创建多个访问路径（如 `shared.weight` 和 `net.2.weight`），导致模型保存后的 `state_dict` 冗余且难以维护。
2.  **初始化陷阱**：使用 `model.apply(init_func)` 时，该共享层会被**重复初始化多次**（本例中为 3 次），这可能无意中覆盖掉你精心设置的特定参数或随机种子。
3.  **钩子冲突**：如果在该层注册了 `forward_hook` 进行调试，它会在一次前向传播中被触发多次，导致数据统计错误。

**规范建议：**
*   **若需参数共享**：推荐在 `__init__` 中只定义一次层属性（如 `self.shared`），然后在 `forward` 函数中手动多次调用它，而**不要**将其塞入 `nn.Sequential` 容器。


---

### 4. 细节补充

关于 `.data` 与 `.detach()` 的区别及用途：

* `.data` (不推荐)

    *   **行为**：它是指向 Tensor 数据的原始指针。
    *   **风险**：修改 `.data` 是**不安全**的。如果你在计算图中修改了它，PyTorch 的自动求导系统（Autograd）无法察觉，这会导致梯度计算错误而不报错。
    *   **用途**：主要用于兼容旧代码。在初始化参数时（如 `param.data.fill_(0)`），因为它不涉及梯度，所以看起来可行，但现代 PyTorch 更推荐在 `torch.no_grad()` 上下文中操作。

* `.detach()` (推荐)

    *   **行为**：返回一个新的 Tensor，从当前计算图中分离出来，但**共享相同的存储空间**。
    *   **安全机制**：它是安全的。如果你修改了 `detach()` 后的 Tensor，而原 Tensor 在反向传播中还需要用到，PyTorch 会报错（计算原 Tensor 的 In-place 修改错误），而不是产生错误的梯度。
    *   **典型用途**：
        *   **截断梯度传播**：在训练 GAN 时固定生成器更新判别器。
        *   **迁移学习**：冻结某些层的权重。
        *   **数值评估**：将输出转为 NumPy (通常用法是 `y.detach().cpu().numpy()`)。

* 总结对比

    | 特性 | `.data` | `.detach()` |
    | :--- | :--- | :--- |
    | **Autograd 监控** | 完全无视，修改可能导致静默的梯度错误 | 监控 In-place 修改，确保梯度计算安全 |
    | **共享内存** | 是 | 是 |
    | **推荐程度** | 不推荐 (Deprecated) | **推荐** |


In [18]:
# 参数提取练习
import torch
from torch import nn

# 创建 3 层 MLP 
net_test1 = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

# 提取并修改第一层偏置为 42.0
with torch.no_grad():
    net_test1[0].bias.fill_(42.0)

# 验证
print(f"第一层偏置: {net_test1[0].bias}")
print(f"验证结果: {'成功' if net_test1[0].bias[0] == 42.0 else '失败'}")

第一层偏置: Parameter containing:
tensor([42., 42., 42., 42., 42., 42., 42., 42.], requires_grad=True)
验证结果: 成功


In [19]:
# 共享参数练习

# 实例化共享参数模型
net_shared = SharedParamMLP(4, 8, 1)

# 模拟一轮训练
X = torch.randn(2, 4)
y = torch.randn(2, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(net_shared.parameters(), lr=0.1)

# 开始训练
net_shared.train()
output = net_shared(X)
loss = loss_fn(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# 验证 Layer 2 和 Layer 4 的参数是否仍指向同一个内存地址
weight2 = net_shared.net[2].weight
weight4 = net_shared.net[4].weight

print(f"物理地址是否相同 (is): {weight2 is weight4}")

# 细节补充：查看它们的梯度，共享层的梯度会是两次调用梯度的累加
print(f"共享层权重的梯度是否存在: {weight2.grad is not None}")

物理地址是否相同 (is): True
共享层权重的梯度是否存在: True


---


## 5.3 延后初始化

> 延后初始化允许我们只定义输出维度，让框架在第一次看到数据时自动推断输入维度。


### 1. 核心动机：为什么需要“懒加载”？

1.  **简化架构设计**：不需要手动计算全连接层的输入维度（特别是接在卷积层后面时）。
2.  **灵活性**：同一个模型定义可以适配不同尺寸的输入（只要逻辑允许）。
3.  **解耦**：模型定义不再强绑定于特定的输入形状。

---

### 2. PyTorch 的实现：`nn.Lazy` 模块

传统的 `nn.Linear` 在定义时就会分配内存并初始化权重。
PyTorch 引入了 **`nn.LazyLinear`**、**`nn.LazyConv2d`** 等模块来实现延后初始化。

In [ ]:
import torch
from torch import nn, Tensor

def deferred_init_demo() -> None:
    """演示 PyTorch 中的延迟初始化 (Lazy Initialization)。"""

    # 推荐实践：仅在 输入维度动态 或 计算复杂的层 使用 Lazy 
    # 后续层建议显式声明维度，以增强代码的可读性和严谨性

    # 1. 定义一个包含 Lazy 层的模型
    # 注意：我们没有指定输入维度 (in_features)
    net = nn.Sequential(
        nn.LazyLinear(256), # 第一层 Lazy：自动适配 X 的特征数
        nn.ReLU(),
        nn.Linear(256, 10) # 后续层显式：此处已知输入必为 256
    )

    # 2. 检查初始化前的状态
    # 此时参数尚未分配内存，打印出来会看到 <uninitialized>
    print(">>> 运行前向传播前的第一层权重: ")
    print(net[0].weight)

    # 3. 模拟输入数据
    X: Tensor = torch.randn(2, 20)

    # 4. 进行第一次前向传播 (触发初始化)
    net(X)

    # 5. 检查初始化后的状态
    print("\n>>> 运行前向传播后的第一层权重形状: ")
    # 框架自动推断出输入是 20，所以形状变为 [256, 20]
    # 权重矩阵 A 的形状是强制转置存储的，即 [in, out]
    print(net[0].weight.shape)
    print(f"权重数值已就绪: {net[0].weight.data.mean():.4f}")

deferred_init_demo()

>>> 运行前向传播前的第一层权重: 
<UninitializedParameter>

>>> 运行前向传播后的第一层权重形状: 
torch.Size([256, 20])
权重数值已就绪: -0.0007


---

### 3. 延后初始化的底层逻辑

当你调用 `net(X)` 时，PyTorch 内部发生了以下动作：

1.  **形状推断**：查看输入 `X` 的最后一个维度（即 20）。
2.  **内存分配**：根据推断出的形状 `(256, 20)` 立即创建权重张量。
3.  **参数初始化**：执行默认的初始化算法（或你通过 `net.apply` 指定的算法）。
4.  **计算流执行**：完成正常的矩阵乘法。

**注意**：一旦初始化完成，该层的形状就固定了。如果第二次传入一个维度不同的 `X`（比如输入维度变成 30），程序会报错。

---

### 4. 补充

#### 4.1 什么时候使用 Lazy 模式？
*   **推荐场景**：当你不想花时间去算 `Flatten` 之后的维度是多少时。
*   **不推荐场景**：如果你需要对模型进行非常精细的**参数初始化控制**（某些初始化算法可能依赖于明确的形状信息），延后初始化可能会让代码逻辑变得隐晦。

#### 4.2 初始化顺序的陷阱
如果你想自定义初始化逻辑，必须在**第一次前向传播之后**进行，或者使用特殊的钩子。

```python
# 错误做法：此时 net[0].weight 还是 uninitialized，调用 apply 会失效或报错
# net.apply(init_weights) 

# 正确做法：
net(X) # 先喂一次数据
net.apply(init_weights) # 再初始化
```

#### 4.3 显存管理
由于参数是延后创建的，这意味着你的显存占用会在训练的第一轮（第一个 Batch）突然跳升。在工程监控中，要留意这个瞬时增长。

---


## 5.4 自定义层

### 1. 不带参数的自定义层

有些层仅仅执行固定的数学变换（如：将输入减去均值），不涉及需要学习的权重。

In [2]:
# 中心化层：用于保证输出均值为 0。

import torch 
from torch import nn, Tensor

class CenteredLayer(nn.Module):
    """自定义不带参数的层：中心化层。
    
    该层在前向传播时将输入减去其均值。
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, X: Tensor) -> Tensor:
        """执行中心化计算。"""
        # 注意：这里不涉及任何 Parameter
        return X - X.mean()
    
# --- 验证 ---
layer = CenteredLayer()
X = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
print(f"中心化输出：{layer(X)}")

中心化输出：tensor([-2., -1.,  0.,  1.,  2.])


---

### 2. 带参数的自定义层

> 如果希望层拥有可以被优化器更新的权重，必须使用 nn.Parameter。

#### 2.1 为什么必须用 `nn.Parameter`？

*   如果你直接写 `self.w = torch.randn(...)`，PyTorch 会认为这只是一个普通的成员变量。
*   当你调用 `net.parameters()` 时，**普通 Tensor 会被忽略**，导致权重无法更新。
*   使用 `nn.Parameter` 封装后，该张量会被自动添加到模型的参数列表中。

#### 2.2 手动实现线性层

In [10]:
class MyLinear(nn.Module):
    """自定义带参数的层：手动实现线性变换。
    
    公式：Y = X @ W.T + b
    """
    def __init__(self, in_units: int, units: int):
        """
        Args:
            in_units: 输入维度。
            units: 输出维度 (神经元数量)。
        """
        super().__init__()

        # 使用 nn.Parameter 注册参数
        # 这样它们就能自动出现在 net.parameters() 中，并支持 to(device) 转移

        # 模仿官方：物理存储为 [out, in]
        self.weight = nn.Parameter(torch.randn(units, in_units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X: Tensor) -> Tensor:
        """执行线性变换。"""
        # 模仿官方：由于存储是反的，计算时要转置回来
        return X @ self.weight.T + self.bias
    
# --- 验证参数注册 ---
dense = MyLinear(5, 3)
print(f"参数列表: \n{list(dense.named_parameters())}") # 看到 'weight'

参数列表: 
[('weight', Parameter containing:
tensor([[ 0.2750,  0.0748, -0.9512, -0.2939,  0.9583],
        [-2.0323, -0.4469, -0.2930,  0.2822, -1.4628],
        [ 0.9947, -0.1764, -1.9233, -1.1410,  0.3239]], requires_grad=True)), ('bias', Parameter containing:
tensor([-0.2154, -1.0346, -0.2718], requires_grad=True))]


---

### 3. 在模型中集成自定义层

> 自定义层与内置层在地位上是平等的，可以无缝嵌入 nn.Sequential。


In [ ]:
def run_custom_layer_demo() -> None:
    """演示如何将自定义层组装进复杂模型。"""

    net = nn.Sequential(
        MyLinear(64, 8),
        nn.ReLU(),
        CenteredLayer(),
        MyLinear(8, 1)
    )

    X = torch.randn(2, 64)
    output = net(X)

    print(f"混合模型输出形状: {output.shape}")
    # 64 * 8 + 8 + 8 * 1 + 1 = 529
    print(f"模型参数总量: {sum(p.numel() for p in net.parameters())}")

run_custom_layer_demo()

混合模型输出形状: torch.Size([2, 1])
模型参数总量: 529


---

### 4. 细节补充

1.  **参数命名**：
    在 `__init__` 中定义的 `nn.Parameter` 变量名（如 `self.weight`）将成为 `named_parameters()` 中的 Key。
2.  **设备一致性**：
    当你调用 `net.to("cuda")` 时，PyTorch 会遍历所有已注册的 `nn.Parameter` 并将它们移到 GPU。如果你手动定义的 Tensor 没用 `Parameter` 包装，这一步会失效。
3.  **初始化习惯**：
    在自定义层中，通常建议在 `__init__` 中加入初始化逻辑（参考 4.8 节），或者在外部使用 `apply`。

---


### 5. 一些练习

In [11]:
# 带权重的中心化层 (简单的标准化为 N(0, 1) 不一定是最合适的，要允许网络找到最适合计算的分布)

# Batch Normalization (BN, 批量归一化) 
# 思想：
#   通过在神经网络层与层之间强行进行“方差调节”和“均值归一”，
#   将中间层输出的分布拉回到标准正态分布附近。
#   再赋予模型自我恢复分布的能力。
# 收益：
#   允许更大的学习率：分布稳定，不再担心梯度爆炸。
#   降低对初始化的敏感度：不再需要极其精细的权重初始化。
#   正则化效果：由于每个 Batch 的均值和方差略有不同，这相当于给网络加了一点点“噪声”，
#             在某些情况下可以起到类似 Dropout 的效果。

class CenteredLayer(nn.Module):
    """自定义带参数的层：带权重的中心化层。"""
    def __init__(self, num_features: int):
        super().__init__()
        
        # 增加可学习的缩放参数 gamma 和偏移参数 beta
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, X: Tensor) -> Tensor:
        """执行带权重的中心化计算。"""
        # 标准实现：计算每个特征列在当前 Batch 上的均值
        # X 形状: [batch_size, num_features]
        # mean 形状: [1, num_features]
        mean = X.mean(dim=0, keepdim=True)

        # 公式: gamma * (X - mean) + beta
        # 利用广播机制，gamma 和 beta 会作用于每一行
        return self.gamma * (X - mean) + self.beta
    
# 维度检查
class MyLinear(nn.Module):
    """自定义带参数的层：手动实现线性变换。
    
    公式：Y = X @ W.T + b
    """
    def __init__(self, in_units: int, units: int):
        """
        Args:
            in_units: 输入维度。
            units: 输出维度 (神经元数量)。
        """
        super().__init__()
        self.in_units = in_units # 保存输入维度用于断言
        self.weight = nn.Parameter(torch.randn(units, in_units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X: Tensor) -> Tensor:
        """执行线性变换。"""
        
        # 维度检查断言
        assert X.shape[-1] == self.in_units, \
            f"输入维度错误! 预期 {self.in_units}, 实际得到 {X.shape[-1]}"

        return X @ self.weight.T + self.bias
    
# --- 测试代码 ---
# 测试 CenteredLayer
layer = CenteredLayer(num_features=5)
X_test = torch.randn(10, 5) + 10  # 均值约等于 10
output = layer(X_test)
print(f"CenteredLayer 输出均值（应接近 0）: {output.mean(dim=0).abs().max().item():.4f}")

# 测试 MyLinear 维度断言
try:
    linear = MyLinear(20, 10)
    linear(torch.randn(2, 19)) # 故意传错维度
except AssertionError as e:
    print(f"断言捕获成功: {e}")


CenteredLayer 输出均值（应接近 0）: 0.0000
断言捕获成功: 输入维度错误! 预期 20, 实际得到 19


---


## 5.5 读写文件


### 1. 读写单个或多个张量

> PyTorch 提供了 torch.save 和 torch.load 两个核心函数。它们内部使用 Python 的 pickle 模块进行序列化。


In [2]:
# 张量持久化

import torch
from torch import Tensor
from pathlib import Path

def tensor_io_demo(save_dir: str = "./temp/data/models") -> None:
    """演示张量的保存与加载。"""
    # 1. 准备存储路径
    path = Path(save_dir)
    path.mkdir(parents=True, exist_ok=True)

    x = torch.arange(4)
    y = torch.zeros(4)

    # 2. 保存单个张量
    # .pt (或 .pth) 是 PyTorch 约定的文件后缀名，是一个二进制文件
    file_single = path / "x.pt"
    torch.save(x, file_single)

    # 3. 保存张量列表或字典 (推荐，更具组织性)
    # 如果有一个参数改变了，一般是重新保存这一组参数。
    file_dict = path / "tensors.pt"
    torch.save({'x': x, 'y': y}, file_dict)

    # 4. 加载张量
    # weights_only 是安全开关，加载器会限制只解析张量、字典和基本数据类型，
    # 拒绝执行包含在文件中的任何自定义 Python 代码。
    x_load = torch.load(file_single, weights_only=True)
    dict_load = torch.load(file_dict, weights_only=True)

    print(f"从单文件加载: {x_load}")
    print(f"从字典加载: {dict_load.keys()}")

tensor_io_demo()

从单文件加载: tensor([0, 1, 2, 3])
从字典加载: dict_keys(['x', 'y'])


**工程建议**：
    对于训练日志等需要频繁追加的小数据，建议使用 `TensorBoard` 或简单的 `.txt/.json` 文件；对于模型权重，通常只保存**最新（last.pt）和最好（best.pt）的两个文件**。

> **TensorBoard** 的核心作用是将训练过程中的枯燥数字转化为直观的图表。


---

### 2. 读写模型参数

我们**不保存**整个模型类（因为那样会依赖特定的类定义代码），而是保存模型的 **state_dict** （状态字典）。

> 加载时先在代码里手动实例化模型类，再把参数填进去。



#### 2.1 什么是 `state_dict`？

*   它是一个标准的 Python 字典。
*   它将每一层的名称映射到对应的参数张量（Weight 和 Bias）。
*   **注意**：只有带有参数的层（如 `Linear`, `Conv2d`）才会出现在字典中。


#### 2.2 模型保存与恢复


In [ ]:
from torch import nn

class SimpleMLP(nn.Module):
    """一个简单的用于演示保存逻辑的模型。"""
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, X: Tensor) -> Tensor:
        return self.output(self.relu(self.hidden(X)))
    
def model_io_workflow(model_path: str = "./temp/data/models/mlp.pt") -> None:
    """完整的模型保存与加载工具流。"""
    # 1. 实例化并初始化模型
    net = SimpleMLP()
    X = torch.randn(2, 20)
    Y_pred = net(X) # 记录原始输出用于比对

    # 2. 保存模型参数 (state_dict)
    torch.save(net.state_dict(), model_path)
    print(f"模型参数已保存至: {model_path}")

    # 3. 恢复模型：必须先创建模型实例
    clone_net = SimpleMLP()
    # 将保存的参数加载进实例
    clone_net.load_state_dict(torch.load(model_path, weights_only=True))

    # 4. 验证一致性
    clone_net.eval() # 切换到评估模式
    Y_clone = clone_net(X)

    # 检查两个模型的输出是否完全一致（忽略微小的浮点数漂移）
    is_equal = torch.allclose(Y_pred, Y_clone)
    print(f"模型恢复验证成功: {is_equal}")

model_io_workflow()

模型参数已保存至: ./temp/data/models/mlp.pt
模型恢复验证成功: True


---

### 3. 细节补充

#### 3.1 为什么不直接保存整个模型对象？

*   **路径依赖**：`torch.save(net, path)` 会保存代码的类路径。如果你重构了目录或发给同伴，由于 `import` 路径不一致，加载时会报 `AttributeError` 或 `ModuleNotFoundError`。
*   **安全性风险 (Pickle 反序列化)**：直接保存对象使用的是 Python 的 `pickle`。恶意构造的 `.pt` 文件在 `torch.load` 时可以执行任意系统代码。使用 `state_dict` 配合 `weights_only=True` 是目前官方推荐的安全做法。
*   **灵活性**：只保存参数可以让你在加载时轻松修改模型结构（例如剔除最后分类层进行迁移学习），而保存整个对象则很难修改其内部逻辑。
*   **最佳实践**：只存 `state_dict`，实现“参数”与“代码”分离。

#### 3.2 跨设备加载 (CPU vs. GPU)

这是一个非常容易踩坑的地方。如果你在 GPU 上训练并保存，想在 CPU 上加载：

```python
# 将 GPU 模型加载到 CPU
device = torch.device('cpu')
state_dict = torch.load(model_path, map_location=device)
net.load_state_dict(state_dict)
```

#### 3.3 断点续训 (Checkpoints)

在大型任务中，除了保存权重，通常还要保存**优化器的状态**和**当前 Epoch 数**：

```python
# --- 保存 Checkpoint ---
checkpoint = {
    'epoch': 10,
    'model_state_dict': net.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': current_loss,
}
torch.save(checkpoint, 'checkpoint.pt')

# --- 恢复 Checkpoint ---
checkpoint = torch.load('checkpoint.pt', weights_only=True)
net.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
last_loss = checkpoint['loss']
```

---


### 4. 小练习

In [12]:
# 参数局部修改

import torch
from torch import nn

# 1. 准备模型和原始输出
original_net = nn.Sequential(
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

X = torch.ones(2, 8)
old_output = original_net(X).detach().clone()

# 2. 导出参数字典并手动修改
state_dict = original_net.state_dict()
print(list(state_dict.keys()))
print(f"原始权重:\n{state_dict['0.weight']}")

# 3. 局部修改: 将第一层所有权重置为 1.0
state_dict['0.weight'].fill_(1.0)

# 4. 将修改后的参数加载回网络
original_net.load_state_dict(state_dict)

# 5. 验证输出是否变化
new_output = original_net(X).detach().clone()
print(f"原始输出样例: {old_output[0].item():.4f}")
print(f"修改后输出样例: {new_output[0].item():.4f}")

['0.weight', '0.bias', '2.weight', '2.bias']
原始权重:
tensor([[ 0.1124, -0.2792, -0.3208,  0.0987,  0.0517, -0.1795, -0.2423,  0.3039],
        [ 0.0600, -0.3379, -0.0642, -0.1651, -0.2832,  0.0609, -0.0329, -0.2415],
        [-0.1983, -0.1146, -0.3301, -0.2308,  0.3007,  0.2270,  0.0454,  0.2386],
        [-0.0588,  0.3172, -0.1433,  0.2838, -0.3384,  0.0831,  0.3004,  0.3388]])
原始输出样例: 0.2087
修改后输出样例: 3.7786


In [13]:
# 多模型 “打包” 存储

# 1. 定义两个不同的模型
net_a = nn.Linear(10, 5)
net_b = nn.Sequential(nn.Linear(5, 2), nn.ReLU(), nn.Linear(2, 1))

# 2. 打包保存
multi_model_path: str = "./temp/data/models/multi_models.pt"
torch.save({
    'net_a_state': net_a.state_dict(),
    'net_b_state': net_b.state_dict(),
    'meta': 'Experimental models'
}, multi_model_path)

# 3. 分别恢复
checkpoint = torch.load(multi_model_path, weights_only=True)
new_net_a = nn.Linear(10, 5)
new_net_b = nn.Sequential(nn.Linear(5, 2), nn.ReLU(), nn.Linear(2, 1))

new_net_a.load_state_dict(checkpoint['net_a_state'])
new_net_b.load_state_dict(checkpoint['net_b_state'])

print("成功从单个文件中恢复了两个不同的模型参数！")

成功从单个文件中恢复了两个不同的模型参数！


In [19]:
# 参数复用 (模型截断与合并)
# 这是迁移学习的核心操作：如何提取旧网络的前两层并放入新架构。

# 假设旧网络 (Source)
source_net = nn.Sequential(
    nn.Linear(10, 64), # Layer 0
    nn.ReLU(),         # Layer 1
    nn.Linear(64, 32), # Layer 2
    nn.ReLU(),         # Layer 3
    nn.Linear(32, 10)  # Layer 4
)

# 假设新网络 (Target) 结构完全不同，但前两层维度匹配
target_net = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 128), # 这里开始不一样了
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 32),
    nn.ReLU(),
    nn.Dropout(0.5),         
    nn.Linear(32, 10),
    nn.ReLU(),
    nn.Linear(32, 2)
)

# 核心：利用字典的 update 机制或过滤加载
source_sd = source_net.state_dict()
target_sd = target_net.state_dict()

# 策略：只提取 target_sd 中存在的 key 且形状匹配的参数
# 在 Sequential 中，前两层的 key 通常是 '0.weight', '0.bias'
pretrained_dict = {k:v for k, v in source_sd.items() if k in target_sd and v.size() == target_sd[k].size()}

# 更新并加载
target_sd.update(pretrained_dict)
target_net.load_state_dict(target_sd)

print(f"成功复用的参数键名: {list(pretrained_dict.keys())}")
# 检查：target_net 的前两层参数现在应与 source_net 完全一致
print(f"参数一致性验证: {torch.allclose(target_net[0].weight, source_net[0].weight)}")

# --- 难度升级 ---
# 假设: 想把 source_net 的 Layer 4 (Linear) 移到 target_net 的 Layer 8

source_sd = source_net.state_dict()
target_sd = target_net.state_dict()

# 定义映射关系 {新层 Key: 旧层 Key}
mapping = {
    '8.weight': '4.weight',
    '8.bias': '4.bias'
}

# 构建迁移字典
transfer_dict = {}
for target_key, source_key in mapping.items():
    if source_key in source_sd and target_key in target_sd:
        # 检查尺寸是否匹配
        if source_sd[source_key].size() == target_sd[target_key].size():
            transfer_dict[target_key] = source_sd[source_key]
        else:
            print(f"警告: {source_key} 与 {target_key} 尺寸不匹配")

# 加载与检验
# strict = False 以允许仅加载部分参数
target_net.load_state_dict(transfer_dict, strict=False) 
print(f"成功复用的参数键名: {list(transfer_dict.keys())}")
print(f"参数一致性验证: {torch.allclose(target_net[8].weight, source_net[4].weight)}")

成功复用的参数键名: ['0.weight', '0.bias']
参数一致性验证: True
成功复用的参数键名: ['8.weight', '8.bias']
参数一致性验证: True


#### **⚠️ 重要说明：关于代码规范的注意**

> **注意**：上述代码中 `self.shared` 层既作为独立属性定义，又被放入 `nn.Sequential` 容器的做法，**仅为了直观演示“物理内存共享”这一概念**。在实际生产环境和科研工程中，这种写法被视为 **“Bad Practice”（不佳实践）**。

**原因如下：**
1.  **重复注册与命名混乱**：PyTorch 会为同一个参数创建多个访问路径（如 `shared.weight` 和 `net.2.weight`），导致模型保存后的 `state_dict` 冗余且难以维护。
2.  **初始化陷阱**：使用 `model.apply(init_func)` 时，该共享层会被**重复初始化多次**（本例中为 3 次），这可能无意中覆盖掉你精心设置的特定参数或随机种子。
3.  **钩子冲突**：如果在该层注册了 `forward_hook` 进行调试，它会在一次前向传播中被触发多次，导致数据统计错误。

**规范建议：**
*   **若需参数共享**：推荐在 `__init__` 中只定义一次层属性（如 `self.shared`），然后在 `forward` 函数中手动多次调用它，而**不要**将其塞入 `nn.Sequential` 容器。


---

## 5.6 GPU

> Graphics Processing Unit, 即图形处理器。


### 1. 硬件环境检查

在开始之前，我们需要确认系统中的 NVIDIA 显卡及其驱动是否正常。


#### 1.1 命令行检查


In [ ]:
# 在 Jupyter Notebook 中，以 ! 开头的行表示 Shell 命令。
!nvidia-smi

#### 1.2 PyTorch 运行环境检查


In [ ]:
import torch
from torch import nn, Tensor

def check_gpu_availability() -> None:
    """检查并打印当前系统的 GPU 状态。"""
    # 1. 驱动与 cuda 是否可用 (即是否能使用 GPU 加速)
    print(f"CUDA 是否可用: {torch.cuda.is_available()}")

    # 2. 显卡数量
    gpu_count = torch.cuda.device_count()
    print(f"检测到的 GPU 数量: {gpu_count}")

    # 3. 查看具体显卡型号
    if gpu_count > 0:
        for i in range(gpu_count):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

check_gpu_availability()

### 2. 设备对象

> 在 PyTorch 中，我们使用 torch.device 来代表计算发生的“位置”。


In [ ]:
def get_default_device() -> torch.device:
    """获取当前环境的最佳计算设备 (暂未考虑多个 GPU 情况，即仅简单考虑了硬件类型)。"""
    if torch.cuda.is_available():
        return torch.device("cuda")
        # cuda 与 cuda:0 是等价的
        # 对于第二张显卡，用的是 torch.device("cuda:1")
    
    # 如果是 Mac M1/M2/M3，可以使用 mps
    if torch.mps.is_available():
        return torch.device("mps")
    
    return torch.device("cpu")

device = get_default_device()
print(f"当前使用的设备: {device}")


当前使用的设备: cuda


---

### 3. 张量与 GPU

#### 3.1 在指定设备上创建张量


In [10]:
# 默认创建在 CPU 上
x = torch.ones(2, 3)
print(f"x 所在设备: {x.device}")

# 直接在 GPU 上创建
y = torch.ones(2, 3, device=torch.device('cuda:0'))
print(f"y 所在设备: {y.device}")

x 所在设备: cpu
y 所在设备: cuda:0


#### 3.2 跨设备移动

注意：to(device) 操作是**异步**的，且对于张量来说，它会返回一个**副本**(即不改变原对象，是在显存额外分配内存)。


In [12]:
# device = get_default_device()
x_gpu = x.to(device)
print(f"x 所在设备: {x.device}")
print(f"x 移动后的设备: {x_gpu.device}")

x 所在设备: cpu
x 移动后的设备: cuda:0


---

### 4. 神经网络与 GPU

模型移动到 GPU 的逻辑与张量略有不同：model.to(device) 会**直接修改**原模型（原地操作）。

> 为了保持代码风格的一致性，虽然模型是原地修改，大家通常还是习惯写成：
`net = net.to(device)`


In [13]:
class SimpleNet(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
def run_model_on_gpu() -> None:
    # 1. 实例化模型
    device = get_default_device()
    net = SimpleNet(20, 1)

    # 2. 将整个网络移至 GPU
    # 这一步会递归地将所有参数 (weight/bias) 移至显存
    net = net.to(device) # 也可以仅写成 net.to(device)

    # 3. 准备数据：输入数据也必须在同一台显卡上
    X = torch.randn(32, 20).to(device)

    # 4. 执行计算
    output = net(X)
    print(f"输出结果所在设备: {output.device}")

run_model_on_gpu()

输出结果所在设备: cuda:0


---

### 5. 跨设备陷阱 (The Multi-device Trap)

这是新手最常遇到的报错：
> `RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!`

#### 5.1 为什么会报错？
PyTorch 不支持跨设备的自动计算。如果 `Tensor A` 在 CPU，`Tensor B` 在 GPU，执行 `A + B` 会报错。

#### 5.2 解决方案：设备对齐
在编写自定义层或训练循环时，永远记得检查输入数据的设备：

```python
def training_loop_example(net: nn.Module, batch: tuple[Tensor, Tensor]):
    # 获取模型所在的设备（取第一个参数的设备作为基准）
    dev = next(net.parameters()).device
    
    # 将数据对齐到该设备
    # non_blocking=True 让 CPU 不必等待数据传输完成就继续执行后续逻辑，发挥异步优势。
    X, y = [item.to(dev, non_blocking=True) for item in batch]
    
    # 执行计算...
```

---


### 6. 细节补充

1.  **显存开销**：GPU 显存通常比系统内存小得多（如 8GB vs 32GB）。在训练大模型时，要时刻监控显存，防止 `Out of Memory (OOM)`。
2.  **数据传输瓶颈**：从 CPU (内存) 拷贝数据到 GPU (显存) 是很慢的。
    *   **优化建议**：尽量在一次拷贝中传输大块数据，而不是频繁传输小张量。
3.  **多 GPU 并行**：如果你的 `device_count() > 1`，你可以使用 `cuda:0`, `cuda:1` 分别指定显卡。

---
